- This notebook performs business analysis on the Gold Layer of the FMCG Data Warehouse.
- The analysis uses the Star Schema created in the Gold Layer to generate key business insights such as revenue, customer behavior, product performance, regional sales, and store performance.
- These queries also serve as the basis for the Power BI dashboard.

In [0]:
gold_path= "/Volumes/workspace/default/fmcg_data/gold/"

In [0]:
customers = spark.read.format("delta").load(gold_path + "dim_customers")
products = spark.read.format("delta").load(gold_path + "dim_products")
stores = spark.read.format("delta").load(gold_path + "dim_stores")
time = spark.read.format("delta").load(gold_path + "dim_time")
sales = spark.read.format("delta").load(gold_path + "fact_sales")

In [0]:
# creating views
customers.createOrReplaceTempView("dim_customers")
products.createOrReplaceTempView("dim_products")
stores.createOrReplaceTempView("dim_stores")
time.createOrReplaceTempView("dim_time")
sales.createOrReplaceTempView("fact_sales")

query 1: total revenue generated

In [0]:
%sql

SELECT
ROUND(SUM(Sales_Amount),2) AS Total_Revenue
FROM fact_sales;

Total_Revenue
3.084314898E7


query 2: total orders

In [0]:
%sql
SELECT COUNT(*) AS Total_Orders
FROM fact_sales;

Total_Orders
17000


query 3: total customers

In [0]:
%sql
SELECT COUNT(*) AS Total_Customers
FROM dim_customers;

Total_Customers
850


query 4: revenue by region

In [0]:
%sql
SELECT
s.Region,
ROUND(SUM(f.Sales_Amount),2) AS Revenue
FROM fact_sales f
JOIN dim_stores s
ON f.Store_ID=s.Store_ID
GROUP BY s.Region
ORDER BY Revenue DESC;

Region,Revenue
South,7707505.57
North,7469206.55
East,6417845.34
West,5635804.85
Central,3612786.67


query 5: sales by product category

In [0]:
%sql
SELECT
p.Category,
ROUND(SUM(f.Sales_Amount),2) AS Revenue
FROM fact_sales f
JOIN dim_products p
ON f.Product_ID=p.Product_ID
GROUP BY p.Category
ORDER BY Revenue DESC;

Category,Revenue
Staples,5604123.16
Personal Care,3828210.72
Beverages,3621293.36
Snacks,3513903.73
Frozen Foods,3197025.82
Baby Care,2637801.67
Dairy,2487221.06
Household,1859742.05
Health & Wellness,1456597.79
Home Care,1159156.89


query 6: top 10 products 

In [0]:
%sql
SELECT
p.Product_Name,
ROUND(SUM(f.Sales_Amount),2) AS Revenue
FROM fact_sales f
JOIN dim_products p
ON f.Product_ID=p.Product_ID
GROUP BY p.Product_Name
ORDER BY Revenue DESC
LIMIT 10;

Product_Name,Revenue
Fortune Sunflower Oil 500ml,533156.48
Dove Personal Care 500g,351365.03
Aashirvaad Staple 500g,327610.61
Fortune Staple 1kg,321715.99
Aashirvaad Atta 400g,320161.47
Nescafe Beverage 250g,305156.71
Colgate Toothpaste 2L,302003.74
Nescafe Coffee 200g,298159.98
Safal Green Peas 2L,295401.93
Nescafe Coffee 1kg,293645.24


query 7: monthly sales trend: shows how sales changed monthly

In [0]:
%sql
SELECT t.Year, t.Month_Name, t.Month,
ROUND(SUM(f.Sales_Amount),2) AS Revenue
FROM fact_sales f
JOIN dim_time t
ON DATE(f.Order_Date)=t.Date
GROUP BY
t.Year,
t.Month_Name,
t.Month
ORDER BY
t.Year,
t.Month;

Year,Month_Name,Month,Revenue
2023,January,1,829093.34
2023,February,2,725225.98
2023,March,3,723057.74
2023,April,4,766922.8
2023,May,5,799996.64
2023,June,6,847638.19
2023,July,7,746240.53
2023,August,8,894503.77
2023,September,9,721985.04
2023,October,10,831618.33


query 8: Top Customers: customers generating the highest revenue.

In [0]:
%sql
SELECT
c.Customer_Name,
ROUND(SUM(f.Sales_Amount),2) AS Revenue
FROM fact_sales f
JOIN dim_customers c
ON f.Customer_ID=c.Customer_ID
GROUP BY c.Customer_Name
ORDER BY Revenue DESC
LIMIT 10;

Customer_Name,Revenue
Turvi Sachdev,163498.21
Saksham De,98426.79
Jonathan Venkatesh,93720.53
Indira Murty,93674.29
Anusha Handa,91378.78
Raksha Manne,89762.5
Avi Mitra,89577.78
Suhani Tak,89237.21
Daniel Tak,87141.14
Prisha Devi,85839.37


query 9: Payment Mode Distribution : Analyzes customer payment preferences.

In [0]:
%sql
SELECT
Payment_Mode,
COUNT(*) AS Transactions
FROM fact_sales
GROUP BY Payment_Mode
ORDER BY Transactions DESC;

Payment_Mode,Transactions
Upi,3442
Net Banking,3435
Cash,3410
Card,1757
Credit Card,1678
Wallet,1662
Debit Card,1616


query 10: Store Performance

In [0]:
%sql
SELECT
s.Store_Name,
ROUND(SUM(f.Sales_Amount),2) AS Revenue
FROM fact_sales f
JOIN dim_stores s
ON f.Store_ID=s.Store_ID
GROUP BY s.Store_Name
ORDER BY Revenue DESC;

Store_Name,Revenue
Patna Branch,1764893.9
RetailMart Chennai,1536444.67
RetailMart Jaipur,1522705.85
RetailMart Hyderabad,1455744.09
RetailMart Coimbatore,1446574.1
RetailMart Kolkata,1433047.67
RetailMart Indore,1325231.77
Pune Branch,1187941.86
Delhi Branch,1175207.46
Chandigarh Branch,1116220.09


query 11: revenue by Loyalty Status : Compares revenue generated by customers with different loyalty levels.

In [0]:
%sql
SELECT
c.Loyalty_Status,
ROUND(SUM(f.Sales_Amount),2) AS Revenue
FROM fact_sales f
JOIN dim_customers c
ON f.Customer_ID=c.Customer_ID
GROUP BY c.Loyalty_Status
ORDER BY Revenue DESC;

Loyalty_Status,Revenue
Gold,9651747.47
Silver,9038056.71
Platinum,6921034.35
Vip,2629497.74
Regular,2602812.71


query 12: Average Order Value

In [0]:
%sql
SELECT
ROUND(AVG(Sales_Amount),2) AS Average_Order_Value
FROM fact_sales;

Average_Order_Value
1814.3
